# gwexpy 時刻操作チュートリアル

このノートブックでは、`gwexpy` で拡張された時刻関連の機能について紹介します。主な改善点は以下の通りです：

1. **ベクトル化された時刻変換**: 大量の時刻データを高速に GPS 時刻へ変換できます。
2. **相互運用性の向上**: Pandas や ObsPy の時刻オブジェクトを直接扱えます。
3. **柔軟な `crop` メソッド**: `TimeSeries` などの切り出しに、GPS 秒だけでなく日付文字列や外部オブジェクトが利用可能です。
4. **`as_series` による変換**: 軸（`gwpy.types.index.Index` / 1D `astropy.units.Quantity`）を `TimeSeries` / `FrequencySeries` に変換します。

## 1. 高速かつ柔軟な時刻変換 (`to_gps`, `tconvert`)

`gwexpy.time` モジュールは `gwpy.time` を拡張しており、リストや Numpy 配列の一括変換をサポートしています。

In [ ]:
import numpy as np
import pandas as pd

from gwexpy.time import tconvert, to_gps

# 1. 文字列のリストを一括変換
time_list = ["2025-01-01 00:00:00", "2025-01-01 00:01:00", "2025-01-01 00:02:00"]
gps_arr = to_gps(time_list)
print("GPS array:", gps_arr)

# 2. Pandas オブジェクトのサポート
df_times = pd.date_range("2025-01-01", periods=3, freq="h")
gps_from_pd = to_gps(df_times)
print("GPS from Pandas:", gps_from_pd)

# 3. tconvert を使った相互変換 (数値なら datetime へ、文字列なら GPS へ)
now_gps = tconvert("now")
print(f"Current GPS: {now_gps}")

dt_list = tconvert(gps_arr)
print("Datetime back-conversion:", dt_list)

## 2. 柔軟なデータ切り出し (`crop`)

`TimeSeries` などの `crop` メソッドが拡張され、`start` や `end` に直接日付文字列などを指定できるようになりました。

In [ ]:
import matplotlib.pyplot as plt

from gwexpy.plot import Plot
from gwexpy.time import to_gps
from gwexpy.timeseries import TimeSeries

# サンプルデータの作成 (2025-01-01 00:00:00 UTC)
t0 = to_gps("2025-01-01 00:00:00")
data = np.random.randn(3600)
ts = TimeSeries(data, t0=t0, dt=1)

print(f"Original span: {ts.span}")

# GPS 秒の代わりに文字列で切り出し
ts_cropped = ts.crop(start="2025-01-01 00:10:00", end="2025-01-01 00:20:00")

print(f"Cropped span: {ts_cropped.span}")
print(f"Number of samples: {len(ts_cropped)}")
Plot(ts, ts_cropped, alpha=0.8)
plt.legend(["original", "cropped"])
plt.show()

## 3. シリーズオブジェクトの自動生成 (`as_series`)

`as_series` 関数は、1D の軸（`gwpy.types.index.Index` または `astropy.units.Quantity`）を `TimeSeries` / `FrequencySeries` に変換します（値は軸の値を使う identity 変換）。

In [ ]:
from gwexpy.types import as_series

# Astropy Quantity からの変換と自動単位変換
times = ts.times
ts_from_times = as_series(times, unit="h")  # s から h へ自動変換して生成
print(type(times))
print(ts_from_times)
ts_from_times.plot()
plt.show()
plt.close()

frequencies = ts.asd(64, 32).frequencies
fs_from_frequencies = as_series(
    frequencies, unit="mHz"
)  # Hz から mHz へ自動変換して生成
print(type(frequencies))
print(fs_from_frequencies)
fs_from_frequencies.plot()
plt.yscale("linear")
plt.show()
plt.close()